# 02 Train/test split

**Objective:** Define the classification label and split the data into train and test sets. No preprocessing is fitted here, to avoid data leakage.

## 0. Setup and data loading
Imports, constants and loading of the raw dataset.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 3.1.1
NumPy version: 2.5.4


In [2]:
DATA_PATH = "../data/raw/co2_emissions.csv"
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)

print("\nData loaded successfully.")
print("Data shape:", df.shape)


Data loaded successfully.
Data shape: (10000, 20)


## 1. Data preview
First rows of the raw dataset.

In [3]:
print("First 5 rows of the dataset:")
df.head()

First 5 rows of the dataset:


,Body Type,Sex,Diet,How Often Shower,Heating Energy Source,Transport,Vehicle Type,Social Activity,Monthly Grocery Bill,Frequency of Traveling by Air,Vehicle Monthly Distance Km,Waste Bag Size,Waste Bag Weekly Count,How Long TV PC Daily Hour,How Many New Clothes Monthly,How Long Internet Daily Hour,Energy efficiency,Recycling,Cooking_With,CarbonEmission
0,overweight,female,pescatarian,daily,coal,public,NaN,often,230,frequently,210,large,4,7,26,1,No,['Metal'],"['Stove', 'Oven']",2238
1,obese,female,vegetarian,less frequently,natural gas,walk/bicycle,NaN,often,114,rarely,9,extra large,3,9,38,5,No,['Metal'],"['Stove', 'Microwave']",1892
2,overweight,male,omnivore,more frequently,wood,private,petrol,never,138,never,2472,small,1,14,47,6,Sometimes,['Metal'],"['Oven', 'Microwave']",2595
3,overweight,male,omnivore,twice a day,wood,walk/bicycle,NaN,sometimes,157,rarely,74,medium,3,20,5,7,Sometimes,"['Paper', 'Plastic', 'Glass', 'Metal']","['Microwave', 'Grill', 'Airfryer']",1074
4,obese,female,vegetarian,daily,coal,private,diesel,often,266,very frequently,8457,large,1,3,5,6,Yes,['Paper'],['Oven'],4743


## 2. Target definition
Class 1 (`high_emission`) means `CarbonEmission` in the top 20% of the data (0.8 quantile threshold). Features are all columns except the target.

In [4]:
q = 0.8
y_reg = df["CarbonEmission"].copy()
thr = y_reg.quantile(q)
y_clf = (y_reg >= thr).astype(int)

X = df.drop(columns=["CarbonEmission"])
print("Emission threshold:", thr, " | Share of class 1:", y_clf.mean().round(3))

Emission threshold: 2966.0  | Share of class 1: 0.2


## 3. Train/test split
80/20 split, stratified by the classification label so both parts keep the same class share.

In [5]:
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_clf
)

print("Shape of data split.")
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Class 1 share, train:", y_clf_train.mean().round(3), "| test:", y_clf_test.mean().round(3))

Shape of data split.
Train: (8000, 19) Test: (2000, 19)
Class 1 share, train: 0.2 | test: 0.2


## 4. Saving the split
Train and test sets are saved with both targets for `03_modeling`.

In [6]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
print("Save Train/Test split to CSV files.")
train = X_train.assign(CarbonEmission=y_reg_train, high_emission=y_clf_train)
test = X_test.assign(CarbonEmission=y_reg_test, high_emission=y_clf_test)
train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

Save Train/Test split to CSV files.


## 5. Conclusions
- Class 1 means `CarbonEmission` of at least 2966 (the 0.8 quantile), so it makes up 20% of the data.
- The classes are imbalanced (80/20), so the split is stratified and PR-AUC is used instead of accuracy in `03_modeling`.
- The threshold is computed on the full dataset before the split, which is a small simplification.
- The split gives 8,000 train and 2,000 test rows with 19 features, and class 1 is 20% in both parts.
- No preprocessing is fitted here. It is fitted inside the pipelines in `03_modeling`, to avoid data leakage.